# MOMO Colab Quick Trial (초보자 / Beginner)

**한국어**: 로컬 설치 없이 MOMO를 가볍게 체험하는 Colab 노트북입니다. Whisper, Ollama, MOMO CLI를 임시 Google Colab GPU 런타임에서 실행합니다.

**English**: This notebook is the lightweight path for trying MOMO without installing anything locally. It runs Whisper, Ollama, and the MOMO CLI inside a temporary Google Colab GPU runtime.

**한국어**: 위에서 아래로 셀을 하나씩 실행하세요. 민감한 녹화, 긴 회의, 반복 사용은 로컬/서버 GUI를 권장합니다.

**English**: Run the cells one by one from top to bottom. For private recordings, long meetings, or repeated use, use the local/server GUI instead.

## Before running / 실행 전

1. **한국어**: Colab 메뉴에서 **Runtime → Change runtime type → GPU**를 선택하세요.
   **English**: In Colab, choose **Runtime → Change runtime type → GPU**.
2. **한국어**: 셀을 위에서 아래로 하나씩 실행하세요.
   **English**: Run each cell one by one from top to bottom.
3. **한국어**: 안내가 나오면 회의 녹화 파일 하나를 업로드하세요.
   **English**: Upload one meeting recording when prompted.
4. **한국어**: 회의별 요약 조건이 있으면 topic-details 셀을 수정하세요.
   **English**: Edit the topic-details cell if you want meeting-specific focus terms.

**한국어**: CUDA GPU가 없으면 이 노트북은 바로 실패합니다. CPU fallback으로 조용히 느리게 돌지 않습니다.

**English**: The notebook intentionally fails if CUDA is unavailable. It does not silently fall back to CPU.

## Visual guide / 화면 안내

**한국어**: 초보자는 아래 이미지를 보고 그대로 따라가면 됩니다. Colab 화면이 조금 달라도 **Runtime**, **Change runtime type**, **Hardware accelerator**, **GPU**, **Save**라는 단어를 찾으세요.

**English**: Beginners can follow the screenshots below. If the Colab UI looks slightly different, look for **Runtime**, **Change runtime type**, **Hardware accelerator**, **GPU**, and **Save**.

### 1. Runtime 메뉴 열기 / Open Runtime menu

![Open Runtime menu](https://raw.githubusercontent.com/jjunsss/MOMO/main/docs/screenshots/colab-01-runtime-menu.svg)

### 2. Change runtime type 선택 / Choose Change runtime type

![Choose Change runtime type](https://raw.githubusercontent.com/jjunsss/MOMO/main/docs/screenshots/colab-02-change-runtime-type.svg)

### 3. GPU 선택 후 Save / Select GPU and Save

![Select GPU and Save](https://raw.githubusercontent.com/jjunsss/MOMO/main/docs/screenshots/colab-03-select-gpu-save.svg)

### 4. 셀 실행 및 파일 업로드 / Run cells and upload one recording

![Run cells and upload recording](https://raw.githubusercontent.com/jjunsss/MOMO/main/docs/screenshots/colab-04-run-cells-upload.svg)

## 1. Settings / 설정

**한국어**: Colab 체험용 기본값입니다. 빠르게 성공하는 것을 우선하므로 `medium`, `fast`, critique off로 시작합니다. 10분 이상 걸려도 괜찮다면 설정 셀의 고품질 값으로 바꿀 수 있습니다.

**English**: These are trial-stable defaults for Colab. They prioritize a reliable first run: `medium`, `fast`, and critique off. If a 10+ minute run is acceptable, you can switch to the higher-quality values in the settings cell.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time
import urllib.error
import urllib.request
import zipfile

REPO_URL = "https://github.com/jjunsss/MOMO.git"
BRANCH = "main"

# 한국어: 기본값은 빠른 체험용입니다. 10분 이상 걸려도 더 좋은 품질을 원하면
# 아래 값을 ASR_MODEL="large-v3", LLM_NUM_CTX="32768",
# SUMMARY_MODE="thorough", ENABLE_CRITIQUE="true"로 바꾸세요.
# English: Defaults are for a fast trial. For higher quality, if a 10+ minute
# run is acceptable, change the values below to ASR_MODEL="large-v3",
# LLM_NUM_CTX="32768", SUMMARY_MODE="thorough", ENABLE_CRITIQUE="true".
ASR_MODEL = "medium"
LLM_MODEL = "qwen3.5:9b"
LLM_NUM_CTX = "8192"
TORCH_INDEX_URL = "https://download.pytorch.org/whl/cu124"
SUMMARY_MODE = "fast"
ENABLE_CRITIQUE = "false"
OUTPUT_LANGUAGE = "ko"  # "ko" or "en"

PROJECT_DIR = Path("/content/MOMO")
WORKSPACE_DIR = Path("/content/momo_workspace")
VIDEOS_DIR = WORKSPACE_DIR / "videos"
RUNS_DIR = WORKSPACE_DIR / "runs"
OLLAMA_BASE_URL = "http://127.0.0.1:11434"


def run(command, *, cwd=None, env=None):
    if isinstance(command, (list, tuple)):
        printable = " ".join(str(part) for part in command)
    else:
        printable = str(command)
    print(f"$ {printable}", flush=True)
    subprocess.run(command, cwd=cwd, env=env, check=True, shell=isinstance(command, str))


## 2. GPU check / GPU 확인

**한국어**: Colab GPU가 켜져 있는지 확인합니다. 여기서 실패하면 런타임 유형을 GPU로 바꾼 뒤 다시 실행하세요.

**English**: This checks whether the Colab GPU is available. If it fails, switch the runtime type to GPU and rerun.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "Colab GPU is OFF or unavailable. Use Runtime > Change runtime type > GPU, "
        "then restart the runtime and rerun this notebook."
    )

print("CUDA GPU:", torch.cuda.get_device_name(0))
try:
    subprocess.run(["nvidia-smi"], check=False)
except FileNotFoundError:
    print("nvidia-smi is not available, but PyTorch can see CUDA.")


## 3. Install MOMO / MOMO 설치

**한국어**: GitHub에서 MOMO를 받고, ffmpeg와 Python 패키지를 설치합니다. PyTorch는 CUDA 12.4 wheel로 고정합니다.

**English**: This clones MOMO from GitHub and installs ffmpeg plus Python packages. PyTorch is pinned to CUDA 12.4 wheels.

In [ ]:
run("apt-get -qq update")
run("apt-get -qq install -y ffmpeg curl git")

if PROJECT_DIR.exists():
    run(["git", "fetch", "origin", BRANCH], cwd=PROJECT_DIR)
    run(["git", "checkout", BRANCH], cwd=PROJECT_DIR)
    run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=PROJECT_DIR)
else:
    run(["git", "clone", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)])

run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"])
run([sys.executable, "-m", "pip", "install", "-q", "torch", "torchaudio", "--index-url", TORCH_INDEX_URL])
run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[asr]"], cwd=PROJECT_DIR)


## 4. Start Ollama / Ollama 실행

**한국어**: 로컬 LLM 서버인 Ollama를 설치/실행하고 기본 모델을 받습니다. 첫 실행은 몇 분 걸릴 수 있습니다.

**English**: This installs/starts Ollama and pulls the default model. The first run can take several minutes.

In [ ]:
def ollama_ready():
    try:
        with urllib.request.urlopen(f"{OLLAMA_BASE_URL}/api/tags", timeout=2):
            return True
    except (OSError, TimeoutError, urllib.error.URLError):
        return False


if shutil.which("ollama") is None:
    run("curl -fsSL https://ollama.com/install.sh | sh")

if not ollama_ready():
    ollama_env = os.environ.copy()
    ollama_env["OLLAMA_HOST"] = "127.0.0.1:11434"
    log_file = open("/tmp/momo_ollama.log", "ab")
    subprocess.Popen(["ollama", "serve"], stdout=log_file, stderr=subprocess.STDOUT, env=ollama_env)
    for _ in range(90):
        if ollama_ready():
            break
        time.sleep(1)
    else:
        raise RuntimeError("Ollama did not start. Check /tmp/momo_ollama.log, restart the runtime, and rerun.")

run(["ollama", "pull", LLM_MODEL])


## 5. Upload a recording / 녹화 파일 업로드

**한국어**: 회의 녹화 파일 하나를 업로드합니다. 업로드한 파일은 임시 Colab workspace에 저장됩니다.

**English**: Upload one meeting recording. The file is saved in the temporary Colab workspace.

In [ ]:
from google.colab import files

VIDEOS_DIR.mkdir(parents=True, exist_ok=True)
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No recording was uploaded.")

saved_files = []
for name, data in uploaded.items():
    destination = VIDEOS_DIR / Path(name).name
    destination.write_bytes(data)
    saved_files.append(destination)

print("Uploaded:")
for path in saved_files:
    print("-", path)


## 6. Meeting focus / 회의 요약 조건

**한국어**: 요약에서 특히 봐야 할 주제와 조심할 조건을 적습니다. 필요 없으면 기본값 그대로 실행해도 됩니다.

**English**: Add focus topics and caution rules for the summary. You can keep the defaults for a simple trial.

In [ ]:
TOPIC_DETAILS = {
    "title": "Colab MOMO Trial",
    "custom_instruction": "결정사항, 액션 아이템, 리스크, 다음 미팅을 우선 정리해 주세요.",
    "topics": [
        "핵심 논의",
        "결정사항",
        "해야 할 일",
        "다음 미팅",
    ],
    "must_check": [
        "잠정 결정과 확정 결정을 구분한다",
        "기술명, 논문명, 모델명처럼 영어로 말한 고유명사는 영어 원문 표기를 우선한다",
        "언급되지 않은 담당자, 날짜, 다음 회의는 추정하지 않는다",
    ],
    "output_language": OUTPUT_LANGUAGE,
}

WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)
topic_path = WORKSPACE_DIR / "topic_details.json"
topic_path.write_text(json.dumps(TOPIC_DETAILS, ensure_ascii=False, indent=2), encoding="utf-8")
print(topic_path.read_text(encoding="utf-8"))


## 7. Run MOMO / MOMO 실행

**한국어**: 업로드한 녹화에서 오디오를 추출하고, Whisper로 전사한 뒤, LLM으로 요약을 만듭니다.

**English**: This extracts audio from the recording, transcribes it with Whisper, and asks the LLM to write the summary.

In [ ]:
env = os.environ.copy()
env.update(
    {
        "MOMO_ASR_DEVICE": "cuda",
        "MOMO_ASR_MODEL": ASR_MODEL,
        "MOMO_LLM_PROVIDER": "ollama",
        "MOMO_LLM_MODEL": LLM_MODEL,
        "MOMO_LLM_BASE_URL": OLLAMA_BASE_URL,
        "MOMO_LLM_NUM_CTX": LLM_NUM_CTX,
        "MOMO_LLM_REQUEST_TIMEOUT_SECONDS": "1800",
        "MOMO_LLM_SUMMARY_MODE": SUMMARY_MODE,
        "MOMO_ENABLE_CRITIQUE": ENABLE_CRITIQUE,
        "MOMO_OUTPUT_LANGUAGE": OUTPUT_LANGUAGE,
        "OLLAMA_HOST": "127.0.0.1:11434",
    }
)

run(
    [
        sys.executable,
        "-m",
        "meeting_ai.cli",
        "auto",
        "--videos-dir",
        str(VIDEOS_DIR),
        "--runs-dir",
        str(RUNS_DIR),
        "--topic-details",
        str(WORKSPACE_DIR / "topic_details.json"),
        "--asr-model",
        ASR_MODEL,
    ],
    cwd=PROJECT_DIR,
    env=env,
)


## 8. Read and download / 결과 확인 및 다운로드

**한국어**: 최종 Markdown 요약을 화면에 보여주고, 요약/근거/전사 파일을 zip으로 내려받습니다.

**English**: This displays the final Markdown summary and downloads the summary, evidence, and transcript files as a zip.

In [ ]:
from IPython.display import Markdown, display
from google.colab import files

run_dirs = sorted(
    [path for path in RUNS_DIR.iterdir() if path.is_dir()],
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)
if not run_dirs:
    raise RuntimeError("No run output was created.")

latest_run = run_dirs[0]
summary_md = latest_run / "summaries" / "final_summary.md"
if not summary_md.exists():
    raise RuntimeError(f"Final summary was not found: {summary_md}")

print("Latest run:", latest_run)
display(Markdown(summary_md.read_text(encoding="utf-8")))

zip_path = WORKSPACE_DIR / f"{latest_run.name}_momo_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for artifact in [
        summary_md,
        latest_run / "summaries" / "final_summary.json",
        latest_run / "evidence" / "summary_evidence.md",
        latest_run / "transcript" / "normalized_transcript.md",
    ]:
        if artifact.exists():
            archive.write(artifact, artifact.relative_to(latest_run.parent))

files.download(str(zip_path))
